In [9]:
#import libraries
import os
import time
import re
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [11]:
# 1. Cấu hình ban đầu
load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

DATA_DIR = Path("../Data")
OUTPUT_DIR = Path("../Preprocessed_Data")

In [3]:
# Prompt linh hoạt cho nhiều loại tài liệu quy chế
SYSTEM_INSTRUCTION = """
Bạn là chuyên gia tiền xử lý dữ liệu RAG cho hệ thống Quy chế đào tạo ĐHBK Hà Nội.

NHIỆM VỤ:
Phân tích tài liệu PDF hoặc .docx, sửa lỗi trình bày, chuyển bảng sang Markdown, và chia thành các chunk logic.

THÔNG TIN TÀI LIỆU:
- Tên file: {filename}
- parent_doc_id: {doc_id}
- category: {category}
- year: {year}

CHIẾN LƯỢC CHIA CHUNK:
- Đơn vị cơ bản: mỗi Điều = 1 chunk
- Nếu Điều quá ngắn (< 80 từ): gộp với Điều liền kề cùng chủ đề
- Nếu Điều quá dài (> 600 từ): tách theo Khoản, mỗi Khoản là 1 chunk
- Bảng biểu lớn (> 5 hàng): có thể là chunk độc lập
- Phần mở đầu/định nghĩa chung: 1 chunk riêng

QUY TẮC ĐỊNH DẠNG:
1. Mỗi chunk nằm trong <<<CHUNK_START>>> ... <<<CHUNK_END>>>
2. Không trả về bất kỳ lời dẫn, giải thích hay hội thoại nào ngoài các thẻ trên
3. Bảng biểu dùng định dạng Markdown table
4. Hình ảnh/biểu đồ: mô tả bằng [Hình: <nội dung>]
5. Trả lời hoàn toàn bằng tiếng Việt

CẤU TRÚC CHUNK (tuân thủ chính xác, không thêm/bớt field):
<<<CHUNK_START>>>
source: {filename}
parent_doc_id: {doc_id}
chunk_id: {doc_id}_001  ← tăng dần, 3 chữ số, ví dụ _002, _003
chunk_index: 1          ← số nguyên tăng dần
language: vi
category: {category}
year: {year}
chunk_title: [Tên điều khoản hoặc chủ đề chính]
topic_tags: [tag1, tag2, tag3]
summary: [2-3 câu tóm tắt, ưu tiên từ khóa: mã học phần, tín chỉ, cảnh báo học tập, điều kiện xét]

[Nội dung chunk ở đây]
<<<CHUNK_END>>>

VÍ DỤ OUTPUT:
<<<CHUNK_START>>>
source: Ngoai_ngu_2022_Quy_doi_CCTA.pdf
parent_doc_id: Ngoai_ngu_2022_Quy_doi_CCTA
chunk_id: Ngoai_ngu_2022_Quy_doi_CCTA_001
chunk_index: 1
language: vi
category: Ngoai_ngu
year: 2022
chunk_title: Điều 1 – Phạm vi áp dụng
topic_tags: [chứng chỉ ngoại ngữ, quy đổi, điều kiện tốt nghiệp]
summary: Quy định về việc quy đổi chứng chỉ ngoại ngữ quốc tế sang điểm học phần tương đương tại ĐHBK Hà Nội. Áp dụng cho sinh viên đại học chính quy có chứng chỉ IELTS, TOEIC, hoặc tương đương.

Điều 1. Phạm vi áp dụng

Quy định này áp dụng cho sinh viên đại học chính quy của Đại học Bách khoa Hà Nội có nhu cầu quy đổi chứng chỉ ngoại ngữ quốc tế...
<<<CHUNK_END>>>
"""

In [4]:
def upload_and_process(file_path: Path):
    SUPPORTED_EXTENSIONS = {".pdf", ".docx"}

    if not file_path.exists() or file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        print(f"[!] Bỏ qua (không hỗ trợ): {file_path.name}")
        return

    category = file_path.parent.name
    filename = file_path.name
    doc_id = file_path.stem

    year_match = re.search(r'\b(19|20)\d{2}\b', doc_id)
    year = year_match.group() if year_match else "N/A"

    print(f"\n[*] Đang xử lý: {filename} (Category: {category}, Year: {year})")

    try:
        file_upload = client.files.upload(file=file_path)
        while file_upload.state.name == "PROCESSING":
            print(".", end="", flush=True)
            time.sleep(2)
            file_upload = client.files.get(name=file_upload.name)

        if file_upload.state.name == "FAILED":
            print(f"\n[X] Upload thất bại: {filename}")
            return

        formatted_instruction = SYSTEM_INSTRUCTION.format(
            filename=filename,
            doc_id=doc_id,
            category=category,
            year=year
        )

        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=[
                file_upload,
                "Hãy tiền xử lý tài liệu này theo đúng hướng dẫn."
            ],
            config=types.GenerateContentConfig(
                system_instruction=formatted_instruction,
                temperature=0.1
            )
        )

        if "<<<CHUNK_START>>>" not in response.text:
            print(f"\n[!] Output không đúng format — lưu vào thư mục review")
            output_folder = OUTPUT_DIR / "_review" / category
        else:
            output_folder = OUTPUT_DIR / category

        output_folder.mkdir(parents=True, exist_ok=True)
        output_file = output_folder / f"{doc_id}_preprocessed.txt"

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(response.text.strip())

        client.files.delete(name=file_upload.name)
        print(f"\n[V] Xong: {output_file}")

    except Exception as e:
        print(f"\n[X] Lỗi khi xử lý {filename}: {e}")

In [5]:
# --- CHỨC NĂNG 1: Xử lý 1 file cụ thể ---
def process_single_file(path_str: str):
    upload_and_process(Path(path_str))


In [6]:
# --- CHỨC NĂNG 2: Xử lý toàn bộ file trong 1 category ---
def process_category(category_name: str):
    category_path = DATA_DIR / category_name
    if not category_path.is_dir():
        print(f"Lỗi: Category '{category_name}' không tồn tại trong {DATA_DIR}")
        return
    
    files = list(category_path.glob("*.*"))
    print(f"Tìm thấy {len(files)} file trong nhóm {category_name}")
    for file in files:
        upload_and_process(file)

In [7]:
# --- CHỨC NĂNG 3: Xử lý tất cả file trong thư mục Data ---
def process_all():
    total = sum(1 for f in DATA_DIR.rglob("*.pdf"))
    print(f"Bắt đầu xử lý {total} file PDF trong toàn bộ thư mục Data...")
    for category_folder in DATA_DIR.iterdir():
        if category_folder.is_dir():
            process_category(category_folder.name)

In [ ]:
#test
process_single_file(r"C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Data\Ngoai_ngu\Ngoai_ngu_2024_K68.pdf")


[*] Đang xử lý: Ngoai_ngu_2022_Quy_doi_CCTA.pdf (Category: Ngoai_ngu, Year: N/A)

[V] Xong: Preprocessed_Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA_preprocessed.txt
